# Unified CONDOR/FlexDC Training Notebook

This notebook trains the unchanged CONDOR `DataCenterModel` on FlexDC-generated data using one of four target definitions:

1. `condor / normal`
2. `condor / raw`
3. `flexdc / normal`
4. `flexdc / raw`

The same combined FlexDC `grid_search_results.csv` and `grid_search_diagnostics.csv` are used for every variant. The targets are constructed inside `am_unified_training_utilities.py`, so do not create separate cleaned/converted datasets for different model families.

**Cell labels:**
- `[COLAB ONLY]` cells should be run in Google Colab.
- `[LOCAL PC ONLY]` cells are for local Jupyter/VS Code runs, not Colab.
- `[ALL]` cells should be run in either environment after paths are set.


## 0. Choose environment

**Run this cell everywhere.**

Set `RUN_ENV = 'colab'` when using Google Colab. Set `RUN_ENV = 'local_pc'` only if you are running this notebook from your Windows machine with the repo already on disk.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess

RUN_ENV = 'colab'  # 'colab' or 'local_pc'

IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ or Path('/content').exists()
print('Detected Colab-like environment:', IN_COLAB)
print('RUN_ENV:', RUN_ENV)


## 1. [COLAB ONLY] Install minimal dependencies

Run this cell in **Google Colab only**.

Do **not** run this on your local Conda environment unless you intentionally want to install packages into the current notebook kernel. Colab usually already has PyTorch installed, so we only install the smaller packages needed for W&B, data handling, and plots.


In [ ]:
if RUN_ENV == 'colab':
    %pip install -q wandb pandas numpy scikit-learn tqdm matplotlib
else:
    print('SKIP: local_pc mode. Do not pip-install from this notebook.')

import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())


## 2. [COLAB ONLY] Clone or update the GitHub repo

Run this cell in **Google Colab only**.

Opening a notebook from GitHub does **not** make the full repository available in the Colab runtime. You still need to clone the repo here.

Before running, set `COMDER_REPO_URL` to your real GitHub URL. If the repo is private, use the GitHub URL format/token approach you normally use for private Colab clones.


In [ ]:
# ===== EDIT THESE =====
COMDER_REPO_URL = 'PUT_YOUR_COMDER_REPO_URL_HERE'
COMDER_BRANCH = 'main'
# ======================

if RUN_ENV == 'colab':
    if COMDER_REPO_URL.startswith('PUT_'):
        raise ValueError('Set COMDER_REPO_URL to your actual GitHub repo URL before running this cell.')

    WORKSPACE = Path('/content/workspace')
    COMDER_ROOT = WORKSPACE / 'comder-main'
    WORKSPACE.mkdir(parents=True, exist_ok=True)

    if COMDER_ROOT.exists():
        print('Repo already exists. Pulling latest changes...')
        subprocess.run(['git', '-C', str(COMDER_ROOT), 'fetch'], check=True)
        subprocess.run(['git', '-C', str(COMDER_ROOT), 'checkout', COMDER_BRANCH], check=True)
        subprocess.run(['git', '-C', str(COMDER_ROOT), 'pull'], check=True)
    else:
        print('Cloning repo...')
        subprocess.run(['git', 'clone', '--branch', COMDER_BRANCH, COMDER_REPO_URL, str(COMDER_ROOT)], check=True)

    print('COMDER_ROOT:', COMDER_ROOT)
else:
    print('SKIP: local_pc mode. Use the local path setup cell instead.')


## 3. [COLAB OPTIONAL] Copy dataset from Google Drive

Run this cell **only if the new combined dataset is not committed to GitHub**.

Set `USE_GOOGLE_DRIVE_DATA = True` and update the two Drive paths. The cell copies the two combined CSVs into the expected repo location:

`am_flexdc/data/pilots/traditionaliso_newqos_pilot/`

If the dataset is already in the repo, leave this cell as-is and it will skip.


In [ ]:
USE_GOOGLE_DRIVE_DATA = False

# Update these only if USE_GOOGLE_DRIVE_DATA = True
DRIVE_RESULTS_CSV = '/content/drive/MyDrive/path/to/traditional_iso16_newqos_AQA_combined_grid_search_results.csv'
DRIVE_DIAGNOSTICS_CSV = '/content/drive/MyDrive/path/to/traditional_iso16_newqos_AQA_combined_grid_search_diagnostics.csv'

if RUN_ENV == 'colab' and USE_GOOGLE_DRIVE_DATA:
    from google.colab import drive
    drive.mount('/content/drive')

    pilot_dir = COMDER_ROOT / 'am_flexdc' / 'data' / 'pilots' / 'traditionaliso_newqos_pilot'
    pilot_dir.mkdir(parents=True, exist_ok=True)

    subprocess.run(['cp', DRIVE_RESULTS_CSV, str(pilot_dir / 'traditional_iso16_newqos_AQA_combined_grid_search_results.csv')], check=True)
    subprocess.run(['cp', DRIVE_DIAGNOSTICS_CSV, str(pilot_dir / 'traditional_iso16_newqos_AQA_combined_grid_search_diagnostics.csv')], check=True)

    print('Copied dataset into:', pilot_dir)
else:
    print('SKIP: Drive copy not requested.')


## 4. [LOCAL PC ONLY] Set local Windows paths

Run this cell **only if you are using this notebook locally on your PC**.

Do **not** run this in Colab. For Colab, the clone cell above sets `COMDER_ROOT`.


In [ ]:
if RUN_ENV == 'local_pc':
    COMDER_ROOT = Path(r'C:\Users\Achuthan Menon\Desktop\Research Work\comder-main')
    if not COMDER_ROOT.exists():
        raise FileNotFoundError(f'COMDER_ROOT does not exist: {COMDER_ROOT}')
    print('Local COMDER_ROOT:', COMDER_ROOT)
else:
    print('SKIP: Colab mode. COMDER_ROOT should already come from the clone cell.')


## 5. [ALL] Resolve paths and check required files

Run this cell after either the Colab clone path is set or the local PC path is set.

Edit `PILOT_NAME` and `PILOT_PREFIX` if your combined dataset folder or filenames change.


In [ ]:
# Dataset naming from the new pilot combine step
PILOT_NAME = 'traditionaliso_newqos_pilot'
PILOT_PREFIX = 'traditional_iso16_newqos_AQA'

AM_FLEXDC_ROOT = COMDER_ROOT / 'am_flexdc'
TRAIN_DIR = AM_FLEXDC_ROOT / 'train'
MODELS_DIR = AM_FLEXDC_ROOT / 'models'
RESULTS_DIR = AM_FLEXDC_ROOT / 'results' / 'training_runs'
PILOT_DIR = AM_FLEXDC_ROOT / 'data' / 'pilots' / PILOT_NAME

RESULTS_CSV = PILOT_DIR / f'{PILOT_PREFIX}_combined_grid_search_results.csv'
DIAGNOSTICS_CSV = PILOT_DIR / f'{PILOT_PREFIX}_combined_grid_search_diagnostics.csv'

MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

required_paths = [
    TRAIN_DIR / 'data_center_model.py',
    TRAIN_DIR / 'am_unified_training_utilities.py',
    RESULTS_CSV,
    DIAGNOSTICS_CSV,
]
missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    print('Missing required files:')
    for p in missing:
        print(' -', p)
    raise FileNotFoundError('Fix missing paths before training.')

print('Paths OK')
print('TRAIN_DIR:', TRAIN_DIR)
print('RESULTS_CSV:', RESULTS_CSV)
print('DIAGNOSTICS_CSV:', DIAGNOSTICS_CSV)
print('MODELS_DIR:', MODELS_DIR)
print('RESULTS_DIR:', RESULTS_DIR)


## 6. [ALL] Import training code

Run this cell in both Colab and local environments. It adds `am_flexdc/train` to `sys.path` so imports work correctly.


In [ ]:
import os
import sys

os.chdir(TRAIN_DIR)
if str(TRAIN_DIR) not in sys.path:
    sys.path.insert(0, str(TRAIN_DIR))

from data_center_model import DataCenterModel
from am_unified_training_utilities import DCDataset, train_model, evaluate_model

print('Current directory:', Path.cwd())
print('Imports OK')


## 7. [ALL] W&B login

Run this cell if you want W&B logging.

Set `USE_WANDB = False` if you want to train without W&B.


In [ ]:
USE_WANDB = True

if USE_WANDB:
    import wandb
    wandb.login()
else:
    wandb = None
    print('W&B disabled for this notebook run.')


## 8. [ALL] Choose one model variant

Run this cell before training. Change only `TARGET_FAMILY` and `TARGET_MODE` to switch among the four requested models.

Valid combinations:
- `condor / normal`
- `condor / raw`
- `flexdc / normal`
- `flexdc / raw`


In [ ]:
TARGET_FAMILY = 'condor'   # 'condor' or 'flexdc'
TARGET_MODE = 'normal'     # 'normal' or 'raw'

USE_NORM_PR = True
USE_NORM_COST = None       # None = CONDOR uses released scaling; FlexDC stays in logged units
USE_NORM_WLMIX = True
RAW_QOS_AGGREGATION = 'mean'  # 'mean' or 'sum'; confirm with Fatih/Kerim if needed

EPOCHS = 150
LR = 1e-4
BATCH_SIZE = 512
CROSS_VALIDATE = True
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

WANDB_ENTITY = 'amenon06-boston-university'
WANDB_PROJECT = 'flexdc-unified-training'
DATASET_TAG = 'newqos_pilot_v1'
RUN_NAME = f'{TARGET_FAMILY}_{TARGET_MODE}_{DATASET_TAG}'

MODEL_DIR = MODELS_DIR / f'{TARGET_FAMILY}_{TARGET_MODE}'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_FILE = MODEL_DIR / f'am_{TARGET_FAMILY}_{TARGET_MODE}_{DATASET_TAG}_state_dict.pt'
METRICS_FILE = RESULTS_DIR / f'am_{TARGET_FAMILY}_{TARGET_MODE}_{DATASET_TAG}_metrics.csv'

print('Run:', RUN_NAME)
print('Device:', DEVICE)
print('Model will save to:', MODEL_FILE)
print('Metrics will save to:', METRICS_FILE)


## 9. [ALL] Dataset sanity check before training

Run this before training. It verifies that the required columns exist and prints target distributions for the chosen variant.


In [ ]:
dataset = DCDataset(
    results_csv=RESULTS_CSV,
    diagnostics_csv=DIAGNOSTICS_CSV,
    target_family=TARGET_FAMILY,
    target_mode=TARGET_MODE,
    use_norm_pr=USE_NORM_PR,
    use_norm_cost=USE_NORM_COST,
    use_norm_wlmix=USE_NORM_WLMIX,
    raw_qos_aggregation=RAW_QOS_AGGREGATION,
)
dataset.get_statistics()


## 10. [ALL] Train one model

Run this cell to train one selected model variant.

Do **not** run the all-four cell below at the same time unless you intentionally want to train all models.


In [ ]:
config = {
    'target_family': TARGET_FAMILY,
    'target_mode': TARGET_MODE,
    'use_norm_pr': USE_NORM_PR,
    'use_norm_cost': USE_NORM_COST,
    'use_norm_wlmix': USE_NORM_WLMIX,
    'raw_qos_aggregation': RAW_QOS_AGGREGATION,
    'epochs': EPOCHS,
    'lr': LR,
    'batch_size': BATCH_SIZE,
    'cross_validate': CROSS_VALIDATE,
    'device': DEVICE,
    'dataset_tag': DATASET_TAG,
    'results_csv': str(RESULTS_CSV),
    'diagnostics_csv': str(DIAGNOSTICS_CSV),
}

run = None
if USE_WANDB:
    run = wandb.init(
        entity=WANDB_ENTITY,
        project=WANDB_PROJECT,
        name=RUN_NAME,
        config=config,
    )

model = DataCenterModel()
model, train_losses, heldout_losses, target_names = train_model(
    model,
    epochs=EPOCHS,
    lr=LR,
    batch_size=BATCH_SIZE,
    verbose=True,
    cross_validate=CROSS_VALIDATE,
    results_csv=RESULTS_CSV,
    diagnostics_csv=DIAGNOSTICS_CSV,
    target_family=TARGET_FAMILY,
    target_mode=TARGET_MODE,
    use_norm_pr=USE_NORM_PR,
    use_norm_cost=USE_NORM_COST,
    use_norm_wlmix=USE_NORM_WLMIX,
    raw_qos_aggregation=RAW_QOS_AGGREGATION,
    device_name=DEVICE,
    wandb_run=run,
)

torch.save(model.state_dict(), MODEL_FILE)
if run is not None:
    run.save(str(MODEL_FILE))
print('Saved model:', MODEL_FILE)


## 11. [ALL] Evaluate and save metrics

Run this after training one model. It saves a one-row metrics CSV and logs final metrics to W&B if enabled.


In [ ]:
import pandas as pd

y_true_train, y_pred_train, y_true_test, y_pred_test, metrics, target_names = evaluate_model(
    model,
    cross_validate=CROSS_VALIDATE,
    batch_size=BATCH_SIZE,
    results_csv=RESULTS_CSV,
    diagnostics_csv=DIAGNOSTICS_CSV,
    target_family=TARGET_FAMILY,
    target_mode=TARGET_MODE,
    use_norm_pr=USE_NORM_PR,
    use_norm_cost=USE_NORM_COST,
    use_norm_wlmix=USE_NORM_WLMIX,
    raw_qos_aggregation=RAW_QOS_AGGREGATION,
    device_name=DEVICE,
    print_metrics=True,
)

metrics_row = {
    'run_name': RUN_NAME,
    'target_family': TARGET_FAMILY,
    'target_mode': TARGET_MODE,
    'model_file': str(MODEL_FILE),
    'target_names': ','.join(target_names),
    **metrics,
}
metrics_df = pd.DataFrame([metrics_row])
metrics_df.to_csv(METRICS_FILE, index=False)
display(metrics_df.T)

if run is not None:
    run.log(metrics)
    run.summary.update(metrics)
    run.finish()

print('Saved metrics:', METRICS_FILE)


## 12. [ALL / OPTIONAL] Train all four variants

Only run this if you want to train all four requested models in one session.

Leave `RUN_ALL_FOUR = False` unless you are ready. This cell creates four W&B runs and four checkpoints.


In [ ]:
RUN_ALL_FOUR = False

if RUN_ALL_FOUR:
    all_rows = []
    variants = [
        ('condor', 'normal'),
        ('condor', 'raw'),
        ('flexdc', 'normal'),
        ('flexdc', 'raw'),
    ]

    for target_family, target_mode in variants:
        run_name = f'{target_family}_{target_mode}_{DATASET_TAG}'
        model_dir = MODELS_DIR / f'{target_family}_{target_mode}'
        model_dir.mkdir(parents=True, exist_ok=True)
        model_file = model_dir / f'am_{target_family}_{target_mode}_{DATASET_TAG}_state_dict.pt'
        metrics_file = RESULTS_DIR / f'am_{target_family}_{target_mode}_{DATASET_TAG}_metrics.csv'

        config = {
            'target_family': target_family,
            'target_mode': target_mode,
            'use_norm_pr': USE_NORM_PR,
            'use_norm_cost': USE_NORM_COST,
            'use_norm_wlmix': USE_NORM_WLMIX,
            'raw_qos_aggregation': RAW_QOS_AGGREGATION,
            'epochs': EPOCHS,
            'lr': LR,
            'batch_size': BATCH_SIZE,
            'cross_validate': CROSS_VALIDATE,
            'device': DEVICE,
            'dataset_tag': DATASET_TAG,
            'results_csv': str(RESULTS_CSV),
            'diagnostics_csv': str(DIAGNOSTICS_CSV),
        }

        this_run = None
        if USE_WANDB:
            this_run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, name=run_name, config=config)

        this_model = DataCenterModel()
        this_model, train_losses, heldout_losses, target_names = train_model(
            this_model,
            epochs=EPOCHS,
            lr=LR,
            batch_size=BATCH_SIZE,
            verbose=True,
            cross_validate=CROSS_VALIDATE,
            results_csv=RESULTS_CSV,
            diagnostics_csv=DIAGNOSTICS_CSV,
            target_family=target_family,
            target_mode=target_mode,
            use_norm_pr=USE_NORM_PR,
            use_norm_cost=USE_NORM_COST,
            use_norm_wlmix=USE_NORM_WLMIX,
            raw_qos_aggregation=RAW_QOS_AGGREGATION,
            device_name=DEVICE,
            wandb_run=this_run,
        )

        torch.save(this_model.state_dict(), model_file)
        if this_run is not None:
            this_run.save(str(model_file))

        y_true_train, y_pred_train, y_true_test, y_pred_test, metrics, target_names = evaluate_model(
            this_model,
            cross_validate=CROSS_VALIDATE,
            batch_size=BATCH_SIZE,
            results_csv=RESULTS_CSV,
            diagnostics_csv=DIAGNOSTICS_CSV,
            target_family=target_family,
            target_mode=target_mode,
            use_norm_pr=USE_NORM_PR,
            use_norm_cost=USE_NORM_COST,
            use_norm_wlmix=USE_NORM_WLMIX,
            raw_qos_aggregation=RAW_QOS_AGGREGATION,
            device_name=DEVICE,
            print_metrics=True,
        )

        row = {
            'run_name': run_name,
            'target_family': target_family,
            'target_mode': target_mode,
            'model_file': str(model_file),
            'target_names': ','.join(target_names),
            **metrics,
        }
        pd.DataFrame([row]).to_csv(metrics_file, index=False)
        all_rows.append(row)

        if this_run is not None:
            this_run.log(metrics)
            this_run.summary.update(metrics)
            this_run.finish()

    all_metrics = pd.DataFrame(all_rows)
    all_metrics_file = RESULTS_DIR / f'all_four_{DATASET_TAG}_metrics.csv'
    all_metrics.to_csv(all_metrics_file, index=False)
    display(all_metrics)
    print('Saved all metrics:', all_metrics_file)
else:
    print('RUN_ALL_FOUR is False; skipping all-four training.')


## 13. [COLAB OPTIONAL] Save/download outputs

Run this in **Colab only** after training if you want to download the trained checkpoint and metrics.

If you trained all four variants, it is usually easier to zip the whole `models/` and `results/training_runs/` folders.


In [ ]:
if RUN_ENV == 'colab':
    from google.colab import files

    # Download the current single-model checkpoint and metrics.
    if MODEL_FILE.exists():
        files.download(str(MODEL_FILE))
    if METRICS_FILE.exists():
        files.download(str(METRICS_FILE))
else:
    print('SKIP: not running in Colab.')


## 14. [COLAB OPTIONAL] Copy trained outputs to Google Drive

Run this in **Colab only** if you want to persist models to Drive instead of downloading manually.

Set `COPY_OUTPUTS_TO_DRIVE = True` and edit `DRIVE_OUTPUT_DIR`.


In [ ]:
COPY_OUTPUTS_TO_DRIVE = False
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/flexdc_unified_training_outputs'

if RUN_ENV == 'colab' and COPY_OUTPUTS_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    Path(DRIVE_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
    subprocess.run(['cp', '-r', str(MODELS_DIR), DRIVE_OUTPUT_DIR], check=True)
    subprocess.run(['cp', '-r', str(RESULTS_DIR), DRIVE_OUTPUT_DIR], check=True)
    print('Copied outputs to:', DRIVE_OUTPUT_DIR)
else:
    print('SKIP: Drive output copy not requested.')
